In [0]:
dbutils.widgets.text("p_environment", "production")
v_environment = dbutils.widgets.get("p_environment")

In [0]:
dbutils.widgets.text("p_file_date", "2024-12-16")
v_file_date = dbutils.widgets.get("p_file_date")

In [0]:
%run "../Includes/configuration"

In [0]:
%run "../Includes/common_functions"

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType

In [0]:
movie_genre_schema = StructType(fields= [
  StructField('movieId', IntegerType(), False),
  StructField('genreId', IntegerType(), False)
])

In [0]:
movie_genre_df = spark.read \
    .schema(movie_genre_schema) \
    .json(f"{bronze_folder_path}/{v_file_date}/movie_genre.json")

In [0]:
display(movie_genre_df)

movieId,genreId
77174,12
77174,16
77174,35
77174,10751
77332,18
77332,10749
77459,12
77459,14
77459,16
77459,35


In [0]:
from pyspark.sql.functions import col, current_timestamp, lit

In [0]:
movie_genre_renamed_df = add_ingestion_date(movie_genre_df) \
                        .withColumnRenamed('movieId', 'movie_Id') \
                        .withColumnRenamed('genreId', 'genre_Id') \
                        .withColumn('environment', lit(v_environment)) \
                        .withColumn('file_date', lit(v_file_date))

In [0]:
display(movie_genre_renamed_df)

movie_Id,genre_Id,ingestion_date,environment,file_date
77174,12,2026-09-11T04:23:08.384584Z,production,2024-12-30
77174,16,2026-09-11T04:23:08.384584Z,production,2024-12-30
77174,35,2026-09-11T04:23:08.384584Z,production,2024-12-30
77174,10751,2026-09-11T04:23:08.384584Z,production,2024-12-30
77332,18,2026-09-11T04:23:08.384584Z,production,2024-12-30
77332,10749,2026-09-11T04:23:08.384584Z,production,2024-12-30
77459,12,2026-09-11T04:23:08.384584Z,production,2024-12-30
77459,14,2026-09-11T04:23:08.384584Z,production,2024-12-30
77459,16,2026-09-11T04:23:08.384584Z,production,2024-12-30
77459,35,2026-09-11T04:23:08.384584Z,production,2024-12-30


In [0]:
# overwrite_partition("movie_silver", "movie_genres", "file_date", v_file_date)    

In [0]:
merge_delta_lake_2(movie_genre_renamed_df, "movie_silver", "movie_genres", "movie_Id", "genre_Id", "file_date")

In [0]:
# movie_genre_renamed_df.write.mode('append').partitionBy("file_date").format("delta").saveAsTable('movie_silver.movie_genres')

In [0]:
%sql
SELECT file_date, COUNT(1)
FROM movie_silver.movie_genres
GROUP BY file_date;

file_date,count(1)
2024-12-16,7000
2024-12-23,3000
2024-12-30,2160


In [0]:
display(spark.read.table('movie_silver.movie_genres'))

movie_Id,genre_Id,ingestion_date,environment,file_date
5,35,2026-09-11T04:22:20.007089Z,production,2024-12-16
5,80,2026-09-11T04:22:20.007089Z,production,2024-12-16
11,12,2026-09-11T04:22:20.007089Z,production,2024-12-16
11,28,2026-09-11T04:22:20.007089Z,production,2024-12-16
11,878,2026-09-11T04:22:20.007089Z,production,2024-12-16
12,16,2026-09-11T04:22:20.007089Z,production,2024-12-16
12,10751,2026-09-11T04:22:20.007089Z,production,2024-12-16
13,18,2026-09-11T04:22:20.007089Z,production,2024-12-16
13,35,2026-09-11T04:22:20.007089Z,production,2024-12-16
13,10749,2026-09-11T04:22:20.007089Z,production,2024-12-16


In [0]:
%sql
DESCRIBE EXTENDED movie_silver.movie_genres;

col_name,data_type,comment
movie_Id,int,null
genre_Id,int,null
ingestion_date,timestamp,null
environment,string,null
file_date,string,null
# Partition Information,,
# col_name,data_type,comment
file_date,string,null
,,
# Delta Statistics Columns,,


In [0]:
dbutils.notebook.exit("Success")